In [1]:
from pathlib import Path
import uproot
from matplotlib import pyplot as plt
import numpy as np
import mplhep as hep

plt.style.use(hep.style.CMS)
hep.style.use("CMS")

In [2]:
# for year in ("2022", "2023"):
for year in ("2023", ):
    print(f"Year: {year}")
    path_data = Path(f"hists/Histograms_{year}_data.root")
    MC_regions = ["QCD", "TTbar", "VJ", "VV"]
    paths_MC = {
        region: Path(f"hists/Histograms_{year}_MC_{region}.root")
        for region in MC_regions
    }

    # Dictionary to store the combined MC histograms
    combined_hists = {}

    # List of variables we're interested in
    variables = [
        'FatJet1_pt',
        # 'FatJet1_MassSD',
        # 'FatJet1_GloParT_XbbVsQCD',
        # 'FatJet1_Tau3OverTau2'
    ]

    # First, load data histograms
    try:
        data_file = uproot.open(path_data)
    except FileNotFoundError:
        print(f"Skipping {year}")
        continue
    data_hists = {}

    for var in variables:
        # Get the ";1" histogram (assuming this is the one we want)
        data_hists[var] = data_file[f"{var};1"]

    # Now combine MC histograms
    for var in variables:
        first_mc = True
        for region in MC_regions:
            mc_file = uproot.open(paths_MC[region])
            hist = mc_file[f"{var};1"]
            
            if first_mc:
                # Initialize with the first MC histogram
                combined_hists[var] = hist.values()
                first_mc = False
            else:
                # Add subsequent MC histograms
                combined_hists[var] += hist.values()
            print(f"integral({region}): {hist.values().sum()}")

    # Calculate data/MC ratios
    ratios = {}
    for var in variables:
        data_integral = data_hists[var].values().sum()
        mc_integral = combined_hists[var].sum()
        ratios[var] = data_integral / mc_integral

    # Print the ratios
    for var, ratio in ratios.items():
        print(f"{var}: {ratio:.3f}")

Year: 2023
integral(QCD): 601.9762907063705
integral(TTbar): 54822.760044524286
integral(VJ): 5105.788812168612
integral(VV): 320.7797303176485
FatJet1_pt: 0.821


```
Year: 2023
integral(QCD): 545.4580069180811
integral(TTbar): 48326.45946139337
integral(VJ): 3187.2653954063717
integral(VV): 292.6614639621839
FatJet1_pt: 0.954
```